# **PHYSIOLOGICAL SIGNALS PREPROCESSING**
For physiological signals captured by Shimmer3 GSR+, Empatica E4 and EmotiBit devices.

|DEVICE |SIGNAL | UNIT OF MEASUREMENT | SAMPLING FREQUENCY |
|----------|----------|----------|----------|
| SHIMMER3 GSR+	| PPG | [milliVolts](https://shimmersensing.com/wp-content/docs/support/documentation/Optical_Pulse_Sensor_User_Guide_rev1.6.pdf)	| 100.21 Hz |
| |	EDA/GSR	| micro Siemens (µS) | 100.21 Hz|
| EMPATICA E4 |	BVP | [nanoWatt](https://support.empatica.com/hc/en-us/articles/360029719792-E4-data-BVP-expected-signal)	| 64 Hz |
| |	EDA/GSR	| microSiemens (µS) | 4 Hz |
| EMOTIBIT |	PPG (3 channels) | [amount of light](https://www.reddit.com/r/EmotiBit/comments/112ao2g/ppg_raw_units/?rdt=57838) |	25 Hz |
| | EDA/GSR|	microSiemens (µS) |	15 Hz |


The data shall be loaded into the `data` dictionary with the following structure:

```JSON
{
    "empatica": # wearables
        {
            "1":  # participant ID
            {
                "Bvp": pd.DataFrame,
                "Gsr": pd.DataFrame
            },
            "2":
            {
                "Bvp": pd.DataFrame,
                "Gsr": pd.DataFrame
            },
            ... 
        },
    "shimmer": 
    {
        ...
    },
    "emotibit":
    {
        ...
    }
}
```

## Functions and variables definitions

In [1]:
RESAMPLING = True

EDA_FREQ = 15 # Final frequency after resampling
BVP_FREQ = 64

In [2]:

def extract_native_freq(signal, device):
    native_sampling_rates = {
        "emotibit": {"Gsr": 15, "Bvp": 25},
        "shimmer": {"Gsr": 100.21, "Bvp": 100.21},
        "empatica": {"Gsr": 4, "Bvp": 64}
    }

    device_lower = device.lower()
    if "emotibit" in device_lower and signal in native_sampling_rates["emotibit"]:
        return native_sampling_rates["emotibit"][signal]
    elif "shimmer" in device_lower and signal in native_sampling_rates["shimmer"]:
        return native_sampling_rates["shimmer"][signal]
    elif "empatica" in device_lower and signal in native_sampling_rates["empatica"]:
        return native_sampling_rates["empatica"][signal]
    else:
        return None


def extract_new_freq(signal, device):
    new_sampling_rates = {
        "emotibit": {"Gsr": EDA_FREQ, "Bvp": BVP_FREQ},
        "shimmer": {"Gsr": EDA_FREQ, "Bvp": BVP_FREQ},
        "empatica": {"Gsr": EDA_FREQ, "Bvp": BVP_FREQ}
    }

    device_lower = device.lower()
    if "emotibit" in device_lower and signal in new_sampling_rates["emotibit"]:
        return new_sampling_rates["emotibit"][signal]
    elif "shimmer" in device_lower and signal in new_sampling_rates["shimmer"]:
        return new_sampling_rates["shimmer"][signal]
    elif "empatica" in device_lower and signal in new_sampling_rates["empatica"]:
        return new_sampling_rates["empatica"][signal]
    else:
        return None

In [ ]:
import numpy as np
import pandas as pd
from scipy.interpolate import InterpolatedUnivariateSpline


def resample_with_spline(df, target_frequency=15, timestamp_col="Timestamp", signal_col="Gsr"):
    # Ensure order and remove duplicates
    df = df.drop_duplicates(subset=[timestamp_col])

    # Sampling interval in seconds
    target_interval = 1 / target_frequency

    # Timestamps and signal
    timestamps = df[timestamp_col].values
    signal = df[signal_col].values

    # Uniform timestamps for resampling
    new_timestamps = np.arange(
        timestamps.min(), timestamps.max(), target_interval)

    # Cubic spline interpolator
    spline = InterpolatedUnivariateSpline(timestamps, signal, k=3)
    new_signal = spline(new_timestamps)

    # Create resampled DataFrame with the same column names
    resampled_df = pd.DataFrame({
        timestamp_col: new_timestamps,
        signal_col: new_signal
    })

    return resampled_df

## Pipeline


The dictionary structure once the pipeline is finished should be something like this:

```json
{
    "shimmer": {
        "user1": {
            "Gsr": {
                "original": {
                    "df": "DataFrame with original data",
                    "metrics": {
                        "bottcher_quality": "valueX",
                        "kleckner_quality": "valueY",
                        "kleckner_quality_filter": "valueZ"
                    }
                },
                "hampel+IQR": {
                    "original": {
                        "df": "DataFrame without outliers",
                        "metrics": {
                            "bottcher_quality": "valueX1",
                            "kleckner_quality": "valueY1",
                            "kleckner_quality_filter": "valueZ1"
                        }
                    },
                    "butterworth": {
                        "original": {
                            "df": "DataFrame filtered with Butterworth",
                            "metrics": {
                                "bottcher_quality": "valueX2",
                                "kleckner_quality": "valueY2",
                                "kleckner_quality_filter": "valueZ2"
                            }
                        }
                    },
                    "gaussian": {
                        "original": {
                            "df": "DataFrame filtered with Gaussian",
                            "metrics": {
                                "bottcher_quality": "valueX3",
                                "kleckner_quality": "valueY3",
                                "kleckner_quality_filter": "valueZ3"
                            }
                        }
                    }
                }
            },
            "Bvp": {
                "original": {
                    "df": "DataFrame with original data",
                    "metrics": {
                        "skewness": "valueA",
                        "neurokit": "valueB",
                        "bvp_quality": "valueC"
                    }
                },
                "hampel+IQR": {
                    "original": {
                        "df": "DataFrame without outliers",
                        "metrics": {
                            "skewness": "valueA1",
                            "neurokit": "valueB1",
                            "bvp_quality": "valueC1"
                        }
                    },
                    "butterworth": {
                        "original": {
                            "df": "DataFrame filtered with Butterworth",
                            "metrics": {
                                "skewness": "valueA2",
                                "neurokit": "valueB2",
                                "bvp_quality": "valueC2"
                            }
                        }
                    },
                    "fourth_cheby2": {
                        "original": {
                            "df": "DataFrame filtered with Chebyshev II",
                            "metrics": {
                                "skewness": "valueA3",
                                "neurokit": "valueB3",
                                "bvp_quality": "valueC3"
                            }
                        }
                    }
                }
            }
        }
    }
}




### Defining steps

In [4]:
%load_ext autoreload
%autoreload 2

import numpy as np

from utils.metrics import bottcher_quality, kleckner_quality, kleckner_quality_filter
from utils.metrics import skewness, neurokit, maki_quality


class FlexibleDict(dict):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

        for key in list(self.keys()):
            
            if isinstance(key, tuple):
                for sub_key in key:
                    self[sub_key] = self[key]

                del self[key]      
    
    def __getitem__(self, key):
        if key in self:
            return super().__getitem__(key)
        for k in self.keys():
            if k in key:
                return self[k]
        if "other" in self: 
            return self["other"]
        raise KeyError(f"Key {key} not found")


metrics = FlexibleDict({
    "gsr": {
        "bottcher_quality": bottcher_quality, 
        "kleckner_quality": kleckner_quality, 
        "kleckner_quality_filter": kleckner_quality_filter 
        },
    ("bvp", "pi", "pr"): {
        "skewness": skewness,
        "maki_quality": maki_quality
        },
    "other": {
            "no": lambda values, fs: 0
        }
    } 
)

from utils.outliers import hampel_IQR_GSR_BVP, IQR

step1 = {
    "name": "outliers",
    "functions": FlexibleDict({
        "gsr": {"hampel+IQR": hampel_IQR_GSR_BVP,
                },
        ("bvp", "pi", "pr"): {
                "hampel+IQR": hampel_IQR_GSR_BVP,
                },
        "other": {
            "IQR": IQR,
            }
        }
    )
}

class DigitalFilter():

    DIGITAL_FILTER_PI = 3.1415926535897932384626433832795
    DIGITAL_FILTER_E  = 2.7182818284590452353602874713526

    def __init__(self, filtertype, samplingFreq, filterFreq1):

        self._type = filtertype
        self._alpha = pow(self.DIGITAL_FILTER_E, -2.0 * self.DIGITAL_FILTER_PI * filterFreq1 / samplingFreq)
        self._nInitSamples = 0
        self._nPoles = 1
    
    def filter(self, inputSample):
        if self._nInitSamples < self._nPoles:
            self._filteredValue = inputSample
            self._nInitSamples += 1
        
        if self._type == "IIR_LOWPASS":
        
            self._filteredValue = inputSample * (1. - self._alpha) + self._filteredValue * self._alpha
            return self._filteredValue
        
        elif self._type == "IIR_HIGHPASS":
        
            self._filteredValue = inputSample * (1. - self._alpha) + self._filteredValue * self._alpha
            return inputSample - self._filteredValue
        
        else:
            return 0.0


def emotibit_filter(df, column, freq):
    ppgSensorHighpass = DigitalFilter("IIR_HIGHPASS", freq, 1)

    vec_func = np.vectorize(ppgSensorHighpass.filter)
    ppg_filtered = vec_func(df[column].values)

    df[column] = ppg_filtered

    return df

from utils.filtering import four_cheby2_bvp, butterworth_bvp, langevin_bandpass, butterworth_gsr, gaussian_gsr, five_cheby2_gsr

step2 = {
    "name": "filtering",
    "functions": FlexibleDict({
        "gsr": {
            "butterworth": butterworth_gsr,
            "gaussian": gaussian_gsr,
            "5cheby2": five_cheby2_gsr
            },
        ("bvp", "pi", "pr"): {
            "emotibit": emotibit_filter,
            "4cheby2": four_cheby2_bvp,
            "butterworth": butterworth_bvp,
            "langevin": langevin_bandpass,
            },
        "other": {
        }
    }
    )
}

steps = [step1, step2]


In [5]:
%load_ext autoreload
%autoreload 2

import inspect
import os
import pickle

def manage_parameters(func, df, column, freq):
    params = inspect.signature(func).parameters
    if len(params) == 2:
        return func(df, column)
    elif len(params) == 3:
        return func(df, column, freq)
    else:
        raise ValueError(f'Función {func.__name__} con número inesperado de parámetros.')


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [6]:
from tqdm import tqdm

def apply_steps(ref, column, device, steps, current_step=0, extract_freq=extract_native_freq):

    if current_step == len(steps):
        return
    df = ref["original"]["df"].copy()

    functions_to_apply = steps[current_step]["functions"][column.lower()] | steps[current_step]["functions"]["other"] # It is important to apply specific ones and also general ones (others)

    sampling_rate = extract_freq(column, device)

    for func_name, func in functions_to_apply.items():
                    
        ref[func_name] = {"original": {}}

        ref[func_name]["original"]["df"] = manage_parameters(func, df.copy(), column, sampling_rate)
            
        metrics_results = {metric_name: metric_func(ref[func_name]["original"]["df"][column].values, fs=sampling_rate) for metric_name, metric_func in metrics[column.lower()].items()}
        ref[func_name]["original"]["metrics"] = metrics_results

        apply_steps(ref[func_name], column, device, steps, current_step+1, extract_freq=extract_freq)



# Hr

In [7]:
import pandas as pd
import neurokit2 as nk
import numpy as np


class DigitalFilter():

    DIGITAL_FILTER_PI = 3.1415926535897932384626433832795
    DIGITAL_FILTER_E = 2.7182818284590452353602874713526

    def __init__(self, filtertype, samplingFreq, filterFreq1):

        self._type = filtertype
        self._alpha = pow(self.DIGITAL_FILTER_E, -2.0 *
                          self.DIGITAL_FILTER_PI * filterFreq1 / samplingFreq)
        self._nInitSamples = 0
        self._nPoles = 1

    def filter(self, inputSample):
        if self._nInitSamples < self._nPoles:
            self._filteredValue = inputSample
            self._nInitSamples += 1

        if self._type == "IIR_LOWPASS":

            self._filteredValue = inputSample * \
                (1. - self._alpha) + self._filteredValue * self._alpha
            return self._filteredValue

        elif self._type == "IIR_HIGHPASS":

            self._filteredValue = inputSample * \
                (1. - self._alpha) + self._filteredValue * self._alpha
            return inputSample - self._filteredValue

        else:
            return 0.0


def hr_neurokit(df, freq):
    timestamps = df["Timestamp"].values

    hr = nk.ppg_process(df[df.columns[-1]].values, sampling_rate=freq)[
        0]["PPG_Rate"].values
    return pd.DataFrame({"Timestamp": timestamps, "Hr_neurokit": hr})


def hr_emotibit(df, freq):
    timestamps = df["Timestamp"].values
    heartRateFilter = DigitalFilter("IIR_LOWPASS", freq, 1)

    timePeriod = (1.0 / freq) * 1000

    peaks = nk.ppg.ppg_findpeaks(
        df[df.columns[-1]].values, sampling_rate=freq, method="elgendi")["PPG_Peaks"]

    beats = np.zeros(len(df[df.columns[-1]].values), dtype=int)
    beats[peaks] = 1

    heartRate = []
    beat_timestamps = []

    interBeatSampleCount = 0
    for i in range(len(beats)):
        interBeatSampleCount += 1

        if beats[i]:
            interBeatInterval = interBeatSampleCount * timePeriod

            heart_rate = (60.0 / interBeatInterval) * 1000
            heart_rate = heartRateFilter.filter(heart_rate)
            heartRate.append(heart_rate)

            beat_timestamps.append(timestamps[i])

            interBeatSampleCount = 0

    return pd.DataFrame({"Timestamp": beat_timestamps, "Hr_emotibit_e": heartRate})

In [8]:
import copy
import pandas as pd


def calculate_hr(data_pipeline, hr_func, hr_field, extract_freq=extract_native_freq):
    """
    Calcula la frecuencia cardíaca a partir de una señal base (ej. BVP).

    Parameters:
        data_pipeline (dict): Estructura de datos del pipeline.
        signal_name (str): Nombre de la señal base (ej. "Bvp").
        hr_func (callable): Función de procesamiento HR (ej. hr_neurokit).
        hr_field (str): Nombre del campo HR a crear (ej. "Hr_neurokit").
    """

    def process_hr_branch(ref):
        """Procesa recursivamente cada rama del pipeline."""
        if "df" in ref:
            return

        for technique in ref:
            if technique == "original":
                timestamps = ref[technique]["df"]["Timestamp"].values
                try:
                    new_df = hr_func(ref[technique]["df"], freq)
                    ref[technique]["df"] = new_df
                    ref[technique]["metrics"] = {"no": 0}  # No metric para HR
                except Exception as e:
                    print(f"Error calculating {hr_field}: {e}")
                    new_df = pd.DataFrame({
                        "Timestamp": timestamps,
                        # fallback
                        hr_field: [40] * len(timestamps)
                    })
                    ref[technique]["df"] = new_df
                    ref[technique]["metrics"] = {"no": 0}
            else:
                process_hr_branch(ref[technique])

    for device in data_pipeline.keys():
        freq = extract_freq("Bvp", device)
        print(f"Calculating HR for device: {device} with freq: {freq} Hz")

        for user in data_pipeline[device]:
            try:
                data_pipeline[device][user][hr_field] = copy.deepcopy(
                    data_pipeline[device][user]["Bvp"]
                )
            except KeyError:
                continue

            process_hr_branch(data_pipeline[device][user][hr_field])

## Execution

In [9]:
import os
import pickle
from tqdm import tqdm
from utils.loader import load_data

datasets = ["first", "second"]

for dataset_name in datasets:
    print("\n==============================")
    print(f"Starting dataset: {dataset_name}")
    print("==============================\n")

    devices_global, users_list, data, common_columns = load_data(
        f"./data/{dataset_name}")
    print("Loaded dataset.")
    print("Devices:", devices_global)
    print("Users:", users_list)

    # --- Resample signals ---
    print("\nResampling signals...")
    for device in devices_global:
        print(f" Device: {device}")
        for user in users_list:
            print(f"   User: {user}")

            signals_to_process = ["Gsr", "Bvp"]

            for signal in signals_to_process:
                df = data[device][user].get(signal)
                if df is None:
                    print(f"     Signal {signal} missing, skipping.")
                    continue

                native_freq = extract_native_freq(signal, device)
                target_freq = EDA_FREQ if signal == "Gsr" else BVP_FREQ

                print(
                    f"     Signal {signal}: native {native_freq} Hz, target {target_freq} Hz")

                if RESAMPLING and target_freq != native_freq:
                    print(f"     Resampling {signal}...")
                    df = resample_with_spline(
                        df,
                        target_frequency=target_freq,
                        timestamp_col="Timestamp",
                        signal_col=signal
                    )
                else:
                    print(f"     No resampling needed for {signal}")

                data[device][user][signal] = df

    # --- Build data pipeline ---
    print("\nBuilding data pipeline...")
    data_pipeline = {}
    for device in devices_global:
        print(f" Device: {device}")
        if device not in data_pipeline:
            data_pipeline[device] = {}
        for user in data[device]:
            print(f"   User: {user}")
            data_pipeline[device][user] = {}
            for column in data[device][user]:
                print(f"     Column: {column}")

                sampling_rate = extract_new_freq(
                    column, device) if RESAMPLING else extract_native_freq(column, device)

                metrics_results = {
                    metric_name: metric_func(
                        data[device][user][column][column].values,
                        fs=sampling_rate
                    )
                    for metric_name, metric_func in metrics[column.lower()].items()
                }

                data_pipeline[device][user][column] = {
                    "original": {
                        "df": data[device][user][column].copy(),
                        "metrics": metrics_results
                    }
                }

    # --- Apply steps ---
    print("\nApplying steps...")
    for device in tqdm(devices_global):
        print(f"\nDevice: {device}")
        for user in tqdm(data[device]):
            print(f"  User: {user}")
            for column in data[device][user]:
                print(f"    Processing column: {column}")

                apply_steps(
                    data_pipeline[device][user][column],
                    column,
                    device,
                    steps,
                    0,
                    extract_freq=extract_native_freq if not RESAMPLING else extract_new_freq
                )

    # --- Calculate heart rate ---
    print("\nCalculating heart rate...")
    calculate_hr(
        data_pipeline,
        hr_emotibit,
        "Hr",
        extract_freq=extract_native_freq if not RESAMPLING else extract_new_freq
    )
    print("Heart rate calculation done.")

    # --- Save processed data ---
    print("\nSaving processed data...")
    os.makedirs("processed_data", exist_ok=True)
    with open(f"processed_data/processed_data_{dataset_name}.pickle", "wb") as file:
        pickle.dump(data_pipeline, file)
    print(f"Saved to processed_data/processed_data_{dataset_name}.pickle")

    print("\nFinished dataset.\n")


Starting dataset: first

Loaded dataset.
Devices: ['emotibit', 'empatica', 'shimmer']
Users: ['13', '14', '15', '16', '17', '18', '19', '21', '22', '24', '25', '26']

Resampling signals...
 Device: emotibit
   User: 13
     Signal Gsr: native 15 Hz, target 15 Hz
     No resampling needed for Gsr
     Signal Bvp: native 25 Hz, target 64 Hz
     Resampling Bvp...
   User: 14
     Signal Gsr: native 15 Hz, target 15 Hz
     No resampling needed for Gsr
     Signal Bvp: native 25 Hz, target 64 Hz
     Resampling Bvp...
   User: 15
     Signal Gsr: native 15 Hz, target 15 Hz
     No resampling needed for Gsr
     Signal Bvp: native 25 Hz, target 64 Hz
     Resampling Bvp...
   User: 16
     Signal Gsr: native 15 Hz, target 15 Hz
     No resampling needed for Gsr
     Signal Bvp: native 25 Hz, target 64 Hz
     Resampling Bvp...
   User: 17
     Signal Gsr: native 15 Hz, target 15 Hz
     No resampling needed for Gsr
     Signal Bvp: native 25 Hz, target 64 Hz
     Resampling Bvp...
   User

  0%|          | 0/3 [00:00<?, ?it/s]


Device: emotibit


  User: 13
    Processing column: Bvp
    Processing column: Gsr


  User: 14
    Processing column: Bvp
    Processing column: Gsr


...

  User: 7
    Processing column: Bvp
    Processing column: Gsr


  User: 8
    Processing column: Bvp
    Processing column: Gsr


  User: 9
    Processing column: Bvp
    Processing column: Gsr


100%|██████████| 4/4 [07:08<00:00, 107.22s/it]



Calculating heart rate...
Calculating HR for device: emotibit_dorsal with freq: 64 Hz
Error calculating Hr: index 0 is out of bounds for axis 0 with size 0
Error calculating Hr: index 0 is out of bounds for axis 0 with size 0
Error calculating Hr: index 0 is out of bounds for axis 0 with size 0
Error calculating Hr: index 0 is out of bounds for axis 0 with size 0
Error calculating Hr: index 0 is out of bounds for axis 0 with size 0
Error calculating Hr: index 0 is out of bounds for axis 0 with size 0
Error calculating Hr: index 0 is out of bounds for axis 0 with size 0
Error calculating Hr: index 0 is out of bounds for axis 0 with size 0
Error calculating Hr: index 0 is out of bounds for axis 0 with size 0
Error calculating Hr: index 0 is out of bounds for axis 0 with size 0
Error calculating Hr: index 0 is out of bounds for axis 0 with size 0
Error calculating Hr: index 0 is out of bounds for axis 0 with size 0
Calculating HR for device: emotibit_volar with freq: 64 Hz
Error calculat